### Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import sqlite3

from pathlib import Path
from sqlalchemy import create_engine

### Cell 2 — Paths

In [2]:
BASE_DIR = Path.cwd().parent

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(
    exist_ok=True
)

### Cell 3 — Auto Load Everything
No hardcoding.

In [3]:
csv_files = list(
    RAW_DIR.glob("*.csv")
)

datasets = {}

for file in csv_files:

    df = pd.read_csv(file)

    datasets[file.stem] = df

print(
    f"Loaded {len(datasets)} datasets"
)

Loaded 10 datasets


### Cell 4 — Dataset Profiler
This automatically inspects every dataset.

In [5]:
quality = []

for name, df in datasets.items():

    quality.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing": df.isnull().sum().sum(),
        "duplicates": df.duplicated().sum()
    })

quality_df = pd.DataFrame(quality)
quality_df

,dataset,rows,columns,missing,duplicates
0,01_fund_master,40,15,0,0
1,02_nav_history,46000,3,0,0
2,03_aum_by_fund_house,90,5,0,0
3,04_monthly_sip_inflows,48,6,12,0
4,05_category_inflows,144,3,0,0
5,06_industry_folio_count,21,6,0,0
6,07_scheme_performance,40,19,0,0
7,08_investor_transactions,32778,13,0,0
8,09_portfolio_holdings,322,8,0,0
9,10_benchmark_indices,8050,3,0,0


### Cell 5 — Generic Cleaning Class

In [6]:
class Cleaner:

    @staticmethod
    def trim_strings(df):

        obj_cols = df.select_dtypes(
            include="object"
        ).columns

        for col in obj_cols:

            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
            )

        return df

    @staticmethod
    def remove_duplicates(df):

        return df.drop_duplicates()

    @staticmethod
    def standardize_columns(df):

        df.columns = (
            df.columns
            .str.lower()
            .str.strip()
        )

        return df

    @staticmethod
    def fix_missing(df):

        return df.ffill()

    @staticmethod
    def clean(df):
        df = Cleaner.standardize_columns(df)
        df = Cleaner.trim_strings(df)
        df = Cleaner.remove_duplicates(df)

        return df

### Cell 6 — Apply Generic Cleaning

In [8]:
for name, df in datasets.items():

    datasets[name] = Cleaner.clean(df)

print("Generic cleaning completed")

C:\Users\Sai Khedekar\AppData\Local\Temp\ipykernel_476\2200591266.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(
C:\Users\Sai Khedekar\AppData\Local\Temp\ipykernel_476\2200591266.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-stri

Generic cleaning completed


C:\Users\Sai Khedekar\AppData\Local\Temp\ipykernel_476\2200591266.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(
C:\Users\Sai Khedekar\AppData\Local\Temp\ipykernel_476\2200591266.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-stri

### Cell 7 — Dataset Specific Rules
The project specifically requires:

`nav_history`
`investor_transactions`
`scheme_performance`

special cleaning.

### Cell 8 — NAV History Cleaning

In [9]:
nav = datasets["02_nav_history"]

nav["date"] = pd.to_datetime(
    nav["date"]
)

nav = nav.sort_values(
    ["amfi_code","date"]
)

nav["nav"] = (
    nav.groupby("amfi_code")["nav"]
       .ffill()
)

nav = nav[
    nav["nav"] > 0
]

datasets["02_nav_history"] = nav

### Cell 9 — Investor Transactions Cleaning

In [11]:
tx = datasets["08_investor_transactions"]

tx["transaction_type"] = (
    tx["transaction_type"]
    .str.title()
)

tx = tx[
    tx["amount_inr"] > 0
]

tx["transaction_date"] = pd.to_datetime(
    tx["transaction_date"]
)

datasets["08_investor_transactions"] = tx

### Cell 10 — Scheme Performance Cleaning

In [23]:
perf = datasets["07_scheme_performance"]

return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct"
]

for col in return_cols:

    perf[col] = pd.to_numeric(
        perf[col],
        errors="coerce"
    )


# Validate expense ratio
perf["expense_ratio_pct"] = pd.to_numeric(perf["expense_ratio_pct"], errors="coerce")
perf["expense_ratio_flag"] = perf["expense_ratio_pct"].between(0.1, 2.5)
perf.head(20)
datasets["07_scheme_performance"] = perf

### Cell 11 — Save All Cleaned Files Automatically

In [24]:
for name, df in datasets.items():

    output_file = (
        PROCESSED_DIR /
        f"clean_{name}.csv"
    )

    df.to_csv(
        output_file,
        index=False
    )

print(
    "All cleaned files saved"
)

All cleaned files saved


### Cell 12 — Create SQLite Database

In [25]:
engine = create_engine(
    "sqlite:///bluestock_mf.db"
)

### Cell 13 — Auto Load Everything Into SQLite

In [26]:
for file in PROCESSED_DIR.glob("*.csv"):

    table_name = (
        file.stem
        .replace("clean_","")
    )

    df = pd.read_csv(file)

    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False
    )

    print(
        f"Loaded {table_name}"
)

Loaded 01_fund_master
Loaded 02_nav_history
Loaded 03_aum_by_fund_house
Loaded 04_monthly_sip_inflows
Loaded 05_category_inflows
Loaded 06_industry_folio_count
Loaded 07_scheme_performance
Loaded 08_investor_transactions
Loaded 09_portfolio_holdings
Loaded 10_benchmark_indices


### Cell 14 — Row Count Validation

In [27]:
for file in PROCESSED_DIR.glob("*.csv"):

    df = pd.read_csv(file)

    print(
        file.stem,
        len(df)
    )

clean_01_fund_master 40
clean_02_nav_history 46000
clean_03_aum_by_fund_house 90
clean_04_monthly_sip_inflows 48
clean_05_category_inflows 144
clean_06_industry_folio_count 21
clean_07_scheme_performance 40
clean_08_investor_transactions 32778
clean_09_portfolio_holdings 322
clean_10_benchmark_indices 8050


### Cell 15 — Generate Data Dictionary Automatically

In [28]:
with open(
    "data_dictionary.md",
    "w"
) as f:

    for name, df in datasets.items():

        f.write(
            f"\n# {name}\n\n"
        )

        for col in df.columns:

            f.write(
                f"- {col} : {df[col].dtype}\n"
            )

### Cell 16 — SQL Verification

In [29]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(
    query,
    engine
)

,name
0,01_fund_master
1,02_nav_history
2,03_aum_by_fund_house
3,04_monthly_sip_inflows
4,05_category_inflows
5,06_industry_folio_count
6,07_scheme_performance
7,08_investor_transactions
8,09_portfolio_holdings
9,10_benchmark_indices


In [31]:
import pandas as pd
import sqlite3

# 2. Create the database connection
conn = sqlite3.connect('bluestock_mf.db')
conn.close()

print("Data successfully saved to SQLite database!")

Data successfully saved to SQLite database!
